# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print metadata summary (not as dictionary subscripting)
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs (referenced by their `@id`).

In [ ]:
# List all record sets available in the dataset
record_sets = dataset.metadata.recordSets
print("Available Record Sets:")
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] name: {rs.name}, @id: {rs.id}")

# Show fields for each record set
all_rs_fields = {}
for rs in record_sets:
    print(f"\n--- Record Set: {rs.name} (@id: {rs.id}) ---")
    fields = rs.fields
    all_rs_fields[rs.id] = [field.id for field in fields]
    for f in fields:
        print(f"  Field: {f.name}\n    @id: {f.id}\n    Data Type: {f.dataType}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Select a record set to load as example (choose the first one if unsure)
chosen_rs = record_sets[0].id if record_sets else None
if chosen_rs is not None:
    print(f"Loading records from record set: {chosen_rs}")
    records = list(dataset.records(record_set=chosen_rs))
    df = pd.DataFrame(records)
    print("Available columns (fields by @id):", df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field for EDA (example: protein abundance, MW, etc)

import numpy as np
import matplotlib.pyplot as plt

# First, print all columns with their types to help user choose
print("Columns and first row values:")
if not df.empty:
    example_row = df.iloc[0].to_dict()
    for col in df.columns:
        print(f"  {col}: value='{example_row[col]}' type={type(example_row[col])}")
    # Try to infer a numeric column (e.g. ends with '_mw' or represents quantity)
    numeric_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] or df[c].apply(lambda x: isinstance(x, (int, float))).any()]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
    else:
        numeric_field = df.columns[0]
        print(f"No clear numeric columns; using first column as example: {numeric_field}")

    # Attempt to coerce to numeric if needed
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter out NaN
    df_num = df.dropna(subset=[numeric_field])
    threshold = df_num[numeric_field].mean() if df_num.shape[0] > 0 else 0
    filtered_df = df_num[df_num[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field if present
    group_field_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < 10]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df)
    else:
        print("No suitable categorical field for grouping found.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
if not df.empty and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field].dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print("No data to plot.")

## 6. Conclusion
This notebook provided an overview and basic exploratory analysis of the FAIR^2 dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

Key steps included loading the metadata and records using `mlcroissant`, identifying available record sets and fields by their `@id`, extracting data as pandas DataFrames, filtering and normalizing numerical fields, grouping by categorical fields, and visualizing data distributions.

Users can further extend this notebook for domain-specific analysis.

For reproducibility and FAIR-ness: always reference entities in this dataset by their `@id`.